# YOLO11 Training - Road Defect & Pothole Detection

This notebook demonstrates how to validate the dataset, configure hyperparameters, train a YOLO11 model, evaluate it, plot performance curves, and export the model for production.

## Notebook Workflow:
1. **Environment Setup & .env Configuration**
2. **Dataset Validation & Sanity Checks**
3. **Hyperparameter Configuration**
4. **YOLO11 Training**
5. **Validation & Metrics Evaluation**
6. **Test-Set Evaluation**
7. **Confusion Matrix & Curves Generation**
8. **Model Export (ONNX)**
9. **Performance Benchmarking (FPS, Memory, Size)**
10. **Save Final Deliverables to `final_model/`**

## 1. Setup & Configuration

Verify the virtual environment, load variables from `.env`, and import libraries.

In [ ]:
import sys
import os
import shutil
import json
import time
import logging
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ═══════════════════════════════════════════════════════════════════════════════
# VERIFY ENVIRONMENT
# ═══════════════════════════════════════════════════════════════════════════════
python_version = f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}"
python_path = sys.executable
print(f"Using Python: {python_version} from {python_path}")

# Set project root path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / 'backend'))

try:
    from config import config
    print("✓ Backend config loaded successfully.")
    print(f"  ACTIVE_MODEL: {config.ACTIVE_MODEL}")
    print(f"  DEVICE: {config.DEVICE}")
except ImportError:
    print("⚠️ Could not load backend config directly. Will use manual fallback settings.")

In [ ]:
# Import Ultralytics & verify GPU availability
try:
    import torch
    from ultralytics import YOLO
    print("✓ PyTorch version:", torch.__version__)
    print("✓ CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("✓ GPU Device Name:", torch.cuda.get_device_name(0))
    print("✓ Ultralytics YOLO imported successfully.")
except ImportError as e:
    print("Installing dependencies...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ultralytics", "pandas", "matplotlib"])
    import torch
    from ultralytics import YOLO
    print("✓ Setup complete!")

## 2. Dataset Validation & Sanity Checks

Verify the dataset files, count annotations, and check class distributions.

In [ ]:
data_yaml_path = project_root / 'data' / 'data.yaml'
print(f"Checking dataset config: {data_yaml_path}")

if data_yaml_path.exists():
    with open(data_yaml_path, 'r') as f:
        yaml_content = f.read()
    print("\n--- data.yaml Content ---")
    print(yaml_content)
    print("-------------------------")
else:
    print("❌ data.yaml not found!")

# Validate split directories
dataset_dir = project_root / 'dataset' / '02_DETAILED_CRACKS_ANNOTATION'
train_images = dataset_dir / 'images' / 'train'
val_images = dataset_dir / 'images' / 'val'

if train_images.exists():
    train_count = len(list(train_images.glob('*')))
    val_count = len(list(val_images.glob('*')))
    print(f"\n✓ Dataset folders validated:")
    print(f"  - Training images: {train_count}")
    print(f"  - Validation images: {val_count}")
else:
    print(f"❌ Dataset directory not found at: {dataset_dir}")

## 3. Hyperparameter Configuration

Define the parameters we will pass to the YOLO11 model training pipeline.

In [ ]:
model_variant = "yolo11s" # Options: yolo11n, yolo11s, yolo11m
epochs = 50
batch_size = 8
img_size = 640
device = 0 if torch.cuda.is_available() else "cpu"
patience = 15

print("Training Hyperparameters:")
print(f"  - Model: {model_variant}.pt")
print(f"  - Epochs: {epochs}")
print(f"  - Batch Size: {batch_size}")
print(f"  - Device: {device}")
print(f"  - Patience: {patience}")

## 4. YOLO11 Training

Load pretrained weights and begin training on your custom dataset.

In [ ]:
print(f"Loading pretrained {model_variant} model...")
model = YOLO(f"{model_variant}.pt")

print("Starting YOLO11 training run...")
start_time = time.time()
results = model.train(
    data=str(data_yaml_path),
    epochs=epochs,
    imgsz=img_size,
    batch=batch_size,
    device=device,
    patience=patience,
    save=True,
    project=str(project_root / 'runs' / 'base_models'),
    name=f'pothole_detector_{model_variant}',
    workers=2,
    close_mosaic=5,
    plots=True
)
training_duration = time.time() - start_time
print(f"✓ Training completed in {training_duration:.2f} seconds ({training_duration/3600:.2f} hours).")

## 5. Validation & Metrics Evaluation

Validate the model on the verification split.

In [ ]:
print("Running validation split evaluation...")
val_metrics = model.val()

print("Validation Metrics:")
print(f"  - mAP50:       {val_metrics.results_dict['metrics/mAP50(B)']:.4f}")
print(f"  - mAP50-95:    {val_metrics.results_dict['metrics/mAP50-95(B)']:.4f}")
print(f"  - Precision:   {val_metrics.results_dict['metrics/precision(B)']:.4f}")
print(f"  - Recall:      {val_metrics.results_dict['metrics/recall(B)']:.4f}")

## 6. Test-Set Evaluation

Evaluate the model by running inference on a batch of test/validation images.

In [ ]:
test_images_dir = val_images
print(f"Running batch inference on sample validation/test images inside: {test_images_dir}")

test_images = list(test_images_dir.glob('*.jpeg')) + list(test_images_dir.glob('*.jpg'))
if test_images:
    sample_test = test_images[:5]
    for img_path in sample_test:
        preds = model.predict(source=str(img_path), conf=0.25, save=False)
        print(f"  - {img_path.name}: Detected {len(preds[0].boxes)} defects")
else:
    print("⚠️ No test images found.")

## 7. Metrics & Confusion Matrix Plots

Find files containing confusion matrices, PR curves, and plot them here.

In [ ]:
# Find latest training run directory
base_models_dir = project_root / 'runs' / 'base_models'
run_dirs = sorted([d for d in base_models_dir.glob(f'pothole_detector_{model_variant}*') if d.is_dir()],
                  key=os.path.getmtime, reverse=True)

if run_dirs:
    latest_run_dir = run_dirs[0]
    print(f"✓ Found latest training output run: {latest_run_dir}")
    
    # Display confusion matrix
    conf_matrix_path = latest_run_dir / 'confusion_matrix.png'
    if conf_matrix_path.exists():
        print(f"Showing Confusion Matrix: {conf_matrix_path.name}")
        img = plt.imread(str(conf_matrix_path))
        plt.figure(figsize=(8, 8))
        plt.imshow(img)
        plt.axis('off')
        plt.show()
    
    # Display Precision-Recall Curve
    pr_curve_path = latest_run_dir / 'BoxPR_curve.png'
    if pr_curve_path.exists():
        print(f"Showing Precision-Recall Curve: {pr_curve_path.name}")
        img = plt.imread(str(pr_curve_path))
        plt.figure(figsize=(8, 8))
        plt.imshow(img)
        plt.axis('off')
        plt.show()
else:
    print("❌ Run outputs not found.")

## 8. Model Export (ONNX Format)

Export your trained PyTorch model `.pt` file to `ONNX` format.

In [ ]:
print("Exporting model to ONNX format...")
try:
    onnx_path = model.export(format='onnx')
    print(f"✓ Export complete: {onnx_path}")
except Exception as e:
    print(f"❌ Export failed: {e}")

## 9. Performance Benchmarking

Benchmark your trained model's inference speed, FPS, memory usage, and footprint.

In [ ]:
import psutil
print("Benchmarking inference latency and memory footprints...")

# Model size in MB
best_pt = latest_run_dir / 'weights' / 'best.pt'
model_size_mb = best_pt.stat().st_size / (1024 * 1024) if best_pt.exists() else 0.0

# Benchmarking inference speed
inference_times = []
dummy_image = np.zeros((640, 640, 3), dtype=np.uint8)

# Warmup runs
for _ in range(10):
    _ = model.predict(source=dummy_image, verbose=False)

# Measure speed
for _ in range(50):
    t_start = time.time()
    _ = model.predict(source=dummy_image, verbose=False)
    inference_times.append((time.time() - t_start) * 1000)

mean_inference_ms = np.mean(inference_times)
fps = 1000 / mean_inference_ms
cpu_memory_usage_mb = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)

print(f"Benchmark Results:")
print(f"  - Inference Time: {mean_inference_ms:.2f} ms per image")
print(f"  - FPS:            {fps:.1f} frames per second")
print(f"  - Model Size:     {model_size_mb:.2f} MB")
print(f"  - CPU Memory:     {cpu_memory_usage_mb:.2f} MB")

## 10. Save Final Deliverables to `final_model/` folder

Copy all output files, weights, and plots to the structured `final_model/` directory.

In [ ]:
final_model_dir = project_root / 'model_training' / 'final_model'
final_model_dir.mkdir(exist_ok=True)
print(f"Saving final deliverables to: {final_model_dir}")

# 1. Weights best.pt
if best_pt.exists():
    shutil.copy(best_pt, final_model_dir / 'yolo11_pothole.pt')

# 2. results.csv
results_csv = latest_run_dir / 'results.csv'
if results_csv.exists():
    shutil.copy(results_csv, final_model_dir / 'training_metrics.csv')

# 3. Confusion Matrix and Curves
if conf_matrix_path.exists():
    shutil.copy(conf_matrix_path, final_model_dir / 'confusion_matrix.png')
if pr_curve_path.exists():
    shutil.copy(pr_curve_path, final_model_dir / 'pr_curve.png')
f1_curve_path = latest_run_dir / 'BoxF1_curve.png'
if f1_curve_path.exists():
    shutil.copy(f1_curve_path, final_model_dir / 'f1_curve.png')

# 4. Evaluation Report JSON
eval_report = {
    "model_name": f"YOLO11s-Detailed-Cracks",
    "val_metrics": {
        "mAP50": float(val_metrics.results_dict.get('metrics/mAP50(B)', 0)),
        "mAP50_95": float(val_metrics.results_dict.get('metrics/mAP50-95(B)', 0)),
        "precision": float(val_metrics.results_dict.get('metrics/precision(B)', 0)),
        "recall": float(val_metrics.results_dict.get('metrics/recall(B)', 0))
    },
    "benchmark": {
        "inference_time_ms": float(mean_inference_ms),
        "fps": float(fps),
        "model_size_mb": float(model_size_mb),
        "cpu_memory_mb": float(cpu_memory_usage_mb)
    }
}

with open(final_model_dir / 'evaluation_report.json', 'w') as f:
    json.dump(eval_report, f, indent=4)

print("\n✓ All final deliverables copied and saved to final_model/")
print(list(final_model_dir.glob('*')))